In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [2]:

# Loaded raw files

demo = pd.read_csv("/Users/akashkumarsamantray/Customer_experience_project/customer-experience-project/df_final_demo.txt")
experiment = pd.read_csv("/Users/akashkumarsamantray/Customer_experience_project/customer-experience-project/df_final_experiment_clients.txt")
web_1 = pd.read_csv("/Users/akashkumarsamantray/Customer_experience_project/customer-experience-project/df_final_web_data_pt_1.txt")
web_2 = pd.read_csv("/Users/akashkumarsamantray/Customer_experience_project/customer-experience-project/df_final_web_data_pt_2.txt")

# web data comes split into two files (same schema) — combine into one clickstream table
web = pd.concat([web_1, web_2], ignore_index=True)

print("demo:", demo.shape)
print("experiment:", experiment.shape)
print("web (combined):", web.shape)

demo: (70609, 9)
experiment: (70609, 2)
web (combined): (755405, 5)


In [3]:

# Initial inspection (confirm what we found manually earlier)

print(demo.isnull().sum())                                   # ~14-15 rows missing across most columns
print(experiment['Variation'].value_counts(dropna=False))     # ~20k clients have no group label
print(web['process_step'].value_counts())                     # funnel step counts, check drop-off shape

client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64
Variation
Test       26968
Control    23532
NaN        20109
Name: count, dtype: int64
process_step
start      243945
step_1     163193
step_2     133062
step_3     112242
confirm    102963
Name: count, dtype: int64


In [4]:

# Clean demo: drop rows with missing core fields
# Only ~14 rows affected, negligible sample loss — dropping is safer than imputing
# Fabricated demographic values into a small handful of records

demo_clean = demo.dropna().copy()
print(f"Dropped {len(demo) - len(demo_clean)} rows with missing values")

# 'U' (Unknown) and 'X' are real categories, not missing data — keep them,
# but flag for later that 'X' has a very small sample size for segment testing
print(demo_clean['gendr'].value_counts(dropna=False))

Dropped 15 rows with missing values
gendr
U    24122
M    23724
F    22745
X        3
Name: count, dtype: int64


In [5]:

# Clean experiment: drop clients with no assigned Variation
# These clients never entered the labelled Test/Control comparison,
# so they can't contribute to any Test vs Control finding

experiment_clean = experiment.dropna(subset=['Variation']).copy()
print(f"Dropped {len(experiment) - len(experiment_clean)} unassigned clients")

Dropped 20109 unassigned clients


In [6]:

# Clean web: parse dates, remove duplicates, filter to experiment, clients
# Convert string timestamps to real datetime so we can measure time-per-step later

web['date_time'] = pd.to_datetime(web['date_time'])

# Remove exact duplicate event rows (likely logging artefacts)
web_clean = web.drop_duplicates().copy()
print(f"Removed {web.duplicated().sum()} duplicate rows")

# The clickstream contains ~49k clients who aren't part of the labelled experiment at all —
# keeping them would dilute any Test vs Control comparison, so filter them out
web_clean = web_clean[web_clean['client_id'].isin(experiment_clean['client_id'])].copy()
print(f"Web rows after filtering to experiment clients: {len(web_clean)}")

# Sort chronologically within each visit so step sequences are in the right order
# (needed later to detect backtracking/repeated steps, a friction signal)
web_clean = web_clean.sort_values(['client_id', 'visit_id', 'date_time']).reset_index(drop=True)

Removed 10764 duplicate rows
Web rows after filtering to experiment clients: 317235


In [7]:
# Merge demo + experiment into one client-level table
# Web/funnel data is kept separate (event-level) — flattening it to one row per client
# would lose the step sequence the agent needs to investigate later

client_table = experiment_clean.merge(demo_clean, on='client_id', how='inner')
print("Merged client-level table:", client_table.shape)
print(client_table['Variation'].value_counts())

# Sanity check — every web client should now have a matching demo/experiment record
orphans = web_clean[~web_clean['client_id'].isin(client_table['client_id'])]['client_id'].nunique()
print(f"Web clients with no matching demo/experiment record: {orphans}")

Merged client-level table: (50487, 10)
Variation
Test       26961
Control    23526
Name: count, dtype: int64
Web clients with no matching demo/experiment record: 13


In [8]:
# Quick funnel sanity check against known manual finding

# Map steps to an order so we can find each client's furthest point in the funnel

step_order = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
web_clean['step_rank'] = web_clean['process_step'].map(step_order)

furthest_step = (
    web_clean.groupby('client_id')['step_rank']
    .max()
    .reset_index()
    .rename(columns={'step_rank': 'furthest_step_rank'})
)

furthest_step = furthest_step.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')
furthest_step['completed'] = furthest_step['furthest_step_rank'] == 4

# This should land close to the known result: Test ~69%, Control ~66%
# If it's way off, something went wrong in the cleaning/filtering above
completion_by_group = furthest_step.groupby('Variation')['completed'].mean()
print(completion_by_group)

Variation
Control    0.655785
Test       0.692927
Name: completed, dtype: float64


In [9]:
# Save cleaned outputs for the next notebook

client_table.to_csv("client_table_clean.csv", index=False)
web_clean.to_csv("web_events_clean.csv", index=False)
print("Saved client_table_clean.csv and web_events_clean.csv")

Saved client_table_clean.csv and web_events_clean.csv


In [ ]:
## Summary of findings — Data Cleaning & Merge

**Raw data loaded:**
- `demo`: 70,609 rows — client demographics
- `experiment`: 70,609 rows — Test/Control assignment
- `web` (combined): 755,405 rows — clickstream events

**Data quality issues found and handled:**
- `demo`: 15 rows had missing values across nearly every column — dropped (negligible sample loss)
- `demo`: gender field includes `U` (Unknown, 24,122 — the largest group), `M` (23,724), `F` (22,745), and `X` (3) — kept as valid categories, `X` flagged as too small for reliable segment testing later
- `experiment`: 20,109 clients (~28%) had no `Variation` label — dropped, since they weren't part of the labeled Test/Control comparison
- `web`: 10,764 fully duplicate event rows — removed
- `web`: after filtering to only the 50,487 labeled experiment clients, 317,235 event rows remained (down from 755,405) — the majority of raw web traffic belonged to clients outside the labeled experiment

**Final merged client table:** 50,487 clients (Test: 26,961 · Control: 23,526)
- Only 13 web clients had no matching demo/experiment record — negligible, left as an acceptable gap

**Sanity check — completion rate by group:**
| Variation | Completion rate |
|---|---|
| Control | 65.58% |
| Test | 69.29% |

This closely matches the known result from the original manual analysis, confirming the cleaning and merge logic is correct.

**Outputs saved:** `client_table_clean.csv`, `web_events_clean.csv` → used as input for `02_manual_segment_analysis.ipynb`